# Build/update private Kaggle input — AF2 spectral
Jalankan **Runtime → Run all** di Colab akun yang dapat mengakses folder `Coffee_Bean_Detection` di Drive. Notebook ini menjadi satu-satunya jalur import `Drive → Colab → private Kaggle Dataset` untuk Stage-1 spectral. Ketiga D0 di-copy byte-for-byte, diverifikasi SHA/ukuran, lalu **load-tested dengan Ultralytics 8.4.96 sebelum upload**. Tidak melatih dan tidak membuka test.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=True)

import importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/af2-spectral-factorization'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
for attempt in range(3):
    result = subprocess.run([
        'git','clone','--depth','1','--branch',BRANCH,
        'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)
    ])
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 2:
        raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)

subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','kaggle'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO/'src'))
importlib.invalidate_caches()
os.chdir(REPO)
print('REPO COMMIT:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip())

from coffee_detector.drive_project import resolve_drive_project_root
from coffee_detector.experiments.prepare_af2_spectral_kaggle import build_af2_spectral_kaggle_bundle

PROJECT_ROOT = resolve_drive_project_root()
BUNDLE = Path('/content/af2-spectral-kaggle-core-v2')
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
manifest = build_af2_spectral_kaggle_bundle(PROJECT_ROOT, BUNDLE)
assert manifest['test_images_included'] is False
assert manifest['format'] == 'coffee_detector.af2_spectral.kaggle_manifest.v2'
print('PROJECT:', PROJECT_ROOT)
print('BUNDLE:', BUNDLE)
print('D0 LOAD-TEST:')
for name, proof in manifest['checkpoint_validation'].items():
    print(f"  {name}: bytes={proof['bytes']} sha256={proof['sha256']} nc={proof['nc']} params={proof['parameters']} loadable={proof['loadable_by_ultralytics']}")
print('MANIFEST:', manifest['manifest'])


In [ ]:
username = userdata.get('KAGGLE_USERNAME')
key = userdata.get('KAGGLE_KEY')
assert username and key, 'Tambahkan Colab secrets KAGGLE_USERNAME dan KAGGLE_KEY lalu aktifkan notebook access.'
os.environ['KAGGLE_USERNAME'] = username
os.environ['KAGGLE_KEY'] = key

dataset_id = f'{username}/faruq-v3-experiment-core-v1'
metadata = {
    'title': 'Faruq V3 Experiment Core V1',
    'id': dataset_id,
    'licenses': [{'name':'other'}],
    'isPrivate': True,
}
(BUNDLE/'dataset-metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

message = 'AF2 spectral v2: load-tested seed-matched D0 checkpoints and SHA contract'
version_cmd = ['kaggle','datasets','version','-p',str(BUNDLE),'-m',message]
version = subprocess.run(version_cmd, text=True, capture_output=True)
if version.returncode != 0:
    combined = (version.stdout + '\n' + version.stderr).lower()
    missing = any(token in combined for token in ('not found','404','does not exist'))
    if not missing:
        print(version.stdout)
        print(version.stderr)
        raise RuntimeError(f'Kaggle dataset version gagal: returncode={version.returncode}')
    print('Dataset belum ada; membuat private dataset baru...')
    subprocess.run(['kaggle','datasets','create','-p',str(BUNDLE)], check=True)
else:
    print(version.stdout)

print('UPLOAD SELESAI:', f'https://www.kaggle.com/datasets/{dataset_id}')
print('PENTING: di notebook Kaggle buka Input → refresh/update dataset ini ke versi terbaru sebelum Run all.')
print('Stage-1 v2 akan MENOLAK bundle lama yang belum memiliki checkpoint load-test.')
